# Level 3C — Calibration and Model Selection

**Audience:** analysts who can simulate return paths and want an auditable way
to estimate parameters and test models out of sample.

**Prerequisites:** Levels 3A–3B, pandas, and basic distribution statistics.

**Learning goals**

1. inspect observed return shape before selecting a model;
2. calibrate GBM, Merton jump diffusion, and Variance Gamma;
3. compare observed and simulated distributions without treating one score as
   proof;
4. run leakage-aware walk-forward validation.

**Outline:** synthetic history → diagnostics → calibration → model comparison
→ walk-forward validation → exercise.

The notebook uses synthetic monthly returns so the complete workflow remains
reproducible and requires no private data or network access.

## 1. Setup and synthetic observed history

We generate one Merton jump-diffusion path and then treat it as if it were an
observed monthly asset history. In real work, replace only this Series while
preserving its frequency and decimal simple-return units.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

pd.options.display.float_format = "{:.4f}".format

from asset_management_toolkit.simulation import (
    calibrate_gbm,
    calibrate_merton_jump,
    calibrate_variance_gamma,
    compare_simulation_models,
    return_distribution_diagnostics,
    simulate_gbm_returns,
    simulate_merton_jump_returns,
    simulate_variance_gamma_returns,
    walk_forward_validate_simulation,
)

In [ ]:
observed = simulate_merton_jump_returns(
    n_years=15,
    n_scenarios=1,
    expected_return=0.07,
    volatility=0.13,
    jump_intensity=1.2,
    jump_mean=-0.10,
    jump_volatility=0.18,
    periods_per_year=12,
    seed=21,
).iloc[:, 0]
observed.index = pd.period_range(
    "2011-01",
    periods=len(observed),
    freq="M",
)
observed.name = "synthetic_asset"
observed.head()

## 2. Diagnose before fitting

Diagnostics describe the sample rather than identify its true data-generating
process. Inspect tail quantiles, skewness, kurtosis, and log-return volatility
before choosing candidate models.

In [ ]:
return_distribution_diagnostics(
    observed,
    periods_per_year=12,
    tail_probability=0.05,
)

## 3. Calibrate three candidate models

- GBM uses closed-form Gaussian log-return maximum likelihood.
- Merton uses deterministic multi-start maximum likelihood over a truncated
  Poisson mixture.
- Variance Gamma matches the second through fourth log-return cumulants.

Because VG does not use a full likelihood here, its AIC/BIC fields are
intentionally empty and must not be invented or compared with likelihood-based
scores.

In [ ]:
gbm_fit = calibrate_gbm(observed, periods_per_year=12)
merton_fit = calibrate_merton_jump(
    observed,
    periods_per_year=12,
    max_jump_intensity=10.0,
)
vg_fit = calibrate_variance_gamma(observed, periods_per_year=12)

calibration_review = pd.DataFrame(
    [fit.to_series() for fit in [gbm_fit, merton_fit, vg_fit]]
).set_index("model")
calibration_review

## 4. Simulate from fitted parameters

Use the same horizon, scenario count, frequency, and seed policy for every
candidate. Different seeds avoid creating artificial path-by-path alignment;
distribution comparison does not require matched random numbers.

In [ ]:
fitted_scenarios = {
    "GBM": simulate_gbm_returns(
        n_years=3,
        n_scenarios=2_000,
        periods_per_year=12,
        seed=101,
        **gbm_fit.parameters,
    ),
    "Merton": simulate_merton_jump_returns(
        n_years=3,
        n_scenarios=2_000,
        periods_per_year=12,
        seed=102,
        **merton_fit.parameters,
    ),
    "Variance Gamma": simulate_variance_gamma_returns(
        n_years=3,
        n_scenarios=2_000,
        periods_per_year=12,
        seed=103,
        **vg_fit.parameters,
    ),
}

In [ ]:
distribution_comparison = compare_simulation_models(
    observed,
    fitted_scenarios,
    periods_per_year=12,
    tail_probability=0.05,
)
distribution_comparison[
    [
        "periodic_mean",
        "periodic_standard_deviation",
        "sample_skewness",
        "sample_excess_kurtosis",
        "q01",
        "q_tail",
        "ks_statistic",
        "wasserstein_distance",
        "distribution_error_score",
    ]
]

## 5. Walk-forward validation

The first 84 months are calibration data and each following 12-month block is
evaluated once. With an expanding window, later folds may use earlier test
periods only after those periods have become historical. No fold sees its own
future returns.

In [ ]:
gbm_walk_forward = walk_forward_validate_simulation(
    observed,
    model="gbm",
    train_size=84,
    test_size=12,
    periods_per_year=12,
    n_scenarios=1_000,
    window="expanding",
    tail_probability=0.05,
    seed=200,
)
gbm_walk_forward[
    [
        "n_train",
        "n_test",
        "mean_error",
        "volatility_error",
        "tail_exceedance_rate",
        "ks_statistic",
        "actual_terminal_return",
        "simulated_terminal_median",
    ]
]

## 6. Exercise — compare out-of-sample models

Run the same walk-forward contract with `model="variance_gamma"`. Then compare:

1. average KS statistic;
2. average Wasserstein distance;
3. tail exceedance rate against the requested 5% tail;
4. simulated median terminal return versus the realized test return.

Do not select a model from one fold or one metric alone.

In [ ]:
vg_walk_forward = walk_forward_validate_simulation(
    observed,
    model="variance_gamma",
    train_size=84,
    test_size=12,
    periods_per_year=12,
    n_scenarios=1_000,
    window="expanding",
    tail_probability=0.05,
    seed=201,
)

### Answer scaffold

In [ ]:
pd.DataFrame(
    {
        "GBM": {
            "mean_ks": gbm_walk_forward["ks_statistic"].mean(),
            "mean_wasserstein": gbm_walk_forward[
                "wasserstein_distance"
            ].mean(),
            "tail_exceedance_rate": (
                gbm_walk_forward["tail_exceedance_rate"].sum()
                / gbm_walk_forward["n_test"].count()
            ),
        },
        "Variance Gamma": {
            "mean_ks": vg_walk_forward["ks_statistic"].mean(),
            "mean_wasserstein": vg_walk_forward[
                "wasserstein_distance"
            ].mean(),
            "tail_exceedance_rate": (
                vg_walk_forward["tail_exceedance_rate"].sum()
                / vg_walk_forward["n_test"].count()
            ),
        },
    }
).T

## Interpretation, pitfalls, and extensions

- Calibration estimates a compact model, not future truth.
- Merton jump parameters can be weakly identified in small samples; inspect
  optimizer status and stability across windows.
- VG cumulant matching is sensitive to sample skewness and kurtosis and does
  not produce likelihood-based AIC/BIC in this toolkit.
- A lower in-sample error does not guarantee better out-of-sample tails.
- Walk-forward folds preserve time order but do not remove regime change,
  transaction costs, liquidity constraints, or model risk.
- Stable-law calibration remains deferred because parameterization and
  heavy-tail estimation need a separate numerical contract.

Next research layers include stress tests, correlated multivariate scenarios,
dynamic allocation, and rates/bond simulation.